Adapted from the book “Hands-On Machine Learning with Scikit-Learn and
TensorFlow” by Aurélien Géron

Adapted from original
[notebook](https://github.com/ageron/handson-ml3/blob/main/02_end_to_end_machine_learning_project.ipynb)
by Aurélien Géron This notebook requires Python 3.7 or later and
scikit-learn 1.0.1 or later.

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.impute import KNNImputer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.feature_selection import SelectFromModel
from pandas.plotting import scatter_matrix

In [ ]:
housing = pd.read_csv("housing.csv")

In [ ]:
# look at the first few rows of the housing dataframe
housing.head()

In [ ]:
# summarize the data
housing.info()

In [ ]:
housing.describe()

In [ ]:
# just one categorical feature
housing["ocean_proximity"].value_counts()

In [ ]:
# Before doing much more, split!
train, test = train_test_split(housing, test_size=0.2, random_state=42)

In [ ]:
# how about a heat map?
housing.plot(
    kind="scatter",
    x="longitude",
    y="latitude",
    grid=True,
    s=housing["population"] / 100,
    label="population",
    c="median_house_value",
    cmap="viridis",
    colorbar=True,
    legend=True,
    figsize=(10, 7),
    alpha=0.2,
)
plt.show()

In [ ]:
# What kinds of scales are we looking at?
scatter_matrix(train, figsize=(12, 12), alpha=0.1)
plt.show()

In [ ]:
train.boxplot(column="median_house_value", by="ocean_proximity")

## Missing features

In [ ]:
# First, let's look at the missing values in the total_bedrooms column
null_rows_idx = train.isnull().any(axis=1)
print(f"Missing {len(train[null_rows_idx])} of {len(train)} rows")
train.loc[null_rows_idx].head()

## Categorical Encoding

In [ ]:
# Option 1: Ordinal encoding
# naive implementation
# If ordinal, needs to make sense - default values are alphabetical
ordinal_encoder = OrdinalEncoder(
    categories=[["ISLAND", "NEAR BAY", "NEAR OCEAN", "<1H OCEAN", "INLAND"]]
)

In [ ]:
# Let's confirm what "near bay" means - San Francisco Bay area?
near_bay = train["ocean_proximity"] == "NEAR BAY"
plt.scatter(train["longitude"], train["latitude"], alpha=0.3)
plt.scatter(train["longitude"][near_bay], train["latitude"][near_bay], alpha=0.3)

## Processing Pipelines

In [ ]:
num_attribs = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
]

# The numeric features get the usual treatment
num_pipeline = make_pipeline(
    KNNImputer(),
    StandardScaler(),
)

# altogether now
preprocessing = make_column_transformer(
    (num_pipeline, num_attribs),
    (ordinal_encoder, ["ocean_proximity"]),
)

preprocessing

In [ ]:
# Finally we can train some kind of model
pipeline = make_pipeline(
    preprocessing,
    LinearRegression(),
)
# cross-validate
ref_mae = cross_val_score(pipeline, train, train["median_house_value"], scoring="neg_mean_absolute_error")
print(-ref_mae.mean())

## A systemic approach for interactions

Baseline: no interactions considered. Then, add on each pairwise
interaction and plot results.

In [ ]:
# 8 numerical attributes plus our single ordinal encoded category
feat_names = num_attribs + ["ocean_proximity"]
k = len(feat_names)
# k choose 2 possible interactions
n_pairs = math.comb(k, 2)
i_effect = np.zeros((n_pairs,))
names = [""] * n_pairs

def interact(X):
    return np.hstack([X, (X[:,0] * X[:,1])[:, np.newaxis]])

count = 0
for i in range(k):
    for j in range(i+1, k):
        pipeline = make_pipeline(
            preprocessing,
            make_column_transformer(
                (FunctionTransformer(interact), [i, j]),
                remainder="passthrough",
            ),
            LinearRegression()
        )
        i_effect[count] = cross_val_score(pipeline, train, train["median_house_value"], scoring="neg_mean_absolute_error").mean()
        names[count] = f"{feat_names[i]} x {feat_names[j]}"
        count = count + 1

In [ ]:
# look at the ones that improve the MAE
order = np.argsort(i_effect, descending=True)
plt.figure(figsize=(12, 4))
plt.scatter(range(n_pairs), -i_effect[order])
plt.hlines(-ref_mae.mean(), 0, n_pairs, colors="red")
plt.text(0, -ref_mae.mean() + 10, "Baseline MAE")
plt.xticks(range(n_pairs), np.asarray(names)[order], rotation=90)
plt.show()

## The easier way

Generate all pairwise interactions, then use feature selection to keep
the ones that improve things. More efficient if your dataset isn’t too
big.

In [ ]:
pipeline = make_pipeline(
    preprocessing,
    PolynomialFeatures(interaction_only=True),
    LinearRegression(),
)
# cross-validate
all_interactions_mae = cross_val_score(pipeline, train, train["median_house_value"], scoring="neg_mean_absolute_error")
-all_interactions_mae.mean()

In [ ]:
pipeline = make_pipeline(
    preprocessing,
    PolynomialFeatures(interaction_only=True),
    SelectFromModel(DecisionTreeRegressor(), threshold="0.75*median"),
    LinearRegression(),
)
# cross-validate
select_interactions_mae = cross_val_score(pipeline, train, train["median_house_value"], scoring="neg_mean_absolute_error")
-select_interactions_mae.mean()

In [ ]:
pipeline[:-1].fit(train, train["median_house_value"])
pipeline[:-1].get_feature_names_out()

## Sanity check: look at your data!

In [ ]:
# Convenient function to evaluate performance
def plot_predictions(actual, pred):
    plt.scatter(actual, pred, alpha=0.1)
    plt.plot([0, 500000], [0, 500000], color="red")
    plt.xlabel("Actual Median House Value ($)")
    plt.ylabel("Predicted Median House Value ($)")
    mae = mean_absolute_error(actual, pred)
    plt.text(300000, 10000, f"MAE = ${mae:.2f}")

In [ ]:
pipeline.fit(train, train["median_house_value"])
y_est = pipeline.predict(train)
plot_predictions(train["median_house_value"], y_est)